In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.preprocessing import StandardScaler
from k_means_constrained import KMeansConstrained

In [2]:
df = pd.read_csv('ipf_reg_2000.csv')
df.head()

,Name,Sex,Event,Equipment,Division,WeightClassKg,Squat1Kg,Bench1Kg,Deadlift1Kg,ProjectedTotal
0,Taylor Atwood,M,SBD,Raw,MR-O,74,255.0,180.0,277.5,712.5
1,Chase Gaddy,M,SBD,Single-ply,M-V,75,217.5,135.0,210.0,562.5
2,Antti Liimatainen,M,SBD,Single-ply,Over 40,83,180.0,130.0,225.0,535.0
3,Keith Bowen,M,SBD,Single-ply,M-O,90,272.5,160.0,280.0,712.5
4,Sophia Ellis,F,SBD,Raw,FR-Jr,84,130.0,92.5,180.0,402.5


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            2000 non-null   object 
 1   Sex             2000 non-null   object 
 2   Event           2000 non-null   object 
 3   Equipment       2000 non-null   object 
 4   Division        2000 non-null   object 
 5   WeightClassKg   2000 non-null   object 
 6   Squat1Kg        2000 non-null   float64
 7   Bench1Kg        2000 non-null   float64
 8   Deadlift1Kg     2000 non-null   float64
 9   ProjectedTotal  2000 non-null   float64
dtypes: float64(4), object(6)
memory usage: 156.4+ KB


In [4]:
df.describe()

,Squat1Kg,Bench1Kg,Deadlift1Kg,ProjectedTotal
count,2000.000000,2000.000000,2000.000000,2000.000000
mean,165.577395,106.582930,183.835470,455.995795
std,59.813516,43.866389,56.110399,154.102727
min,25.000000,25.000000,75.000000,125.000000
25%,120.000000,70.000000,139.375000,330.000000
50%,162.500000,105.000000,185.000000,452.500000
75%,205.000000,135.000000,225.000000,560.000000
max,410.000000,280.000000,350.000000,1020.000000


In [28]:
sample = df.sample(n=42, random_state=42)
sample.head()

,Name,Sex,Event,Equipment,Division,WeightClassKg,Squat1Kg,Bench1Kg,Deadlift1Kg,ProjectedTotal
1860,Elisabeth Castleman,F,SBD,Raw,FR-O,57,80.0,42.5,92.5,215.0
353,Harry Smith,M,SBD,Raw,MR-Sj,74,170.0,130.0,180.0,480.0
1333,Christina Paugh,F,SBD,Raw,FR-O,84+,105.0,57.5,110.0,272.5
905,Magdolna Petróczki,F,SBD,Single-ply,Open,56,145.0,75.0,160.0,380.0
1289,佐藤 駿,M,SBD,Raw,High School,74,160.0,100.0,207.5,467.5


In [27]:
#print(sample.sort_values(by='ProjectedTotal'))
#print(sample.sort_values(by='ProjectedTotal').index.to_list())

For simplicity's sake here, we will use the index value as each lifter's "ID." If this were to be implemented in a real-world case, lifters' federation IDs could be use to address the issue of two lifters having the same name.

In [16]:
# this function creates flights based on the simplest flight-construction method:
#
# 1) sort the df based on projected total
# 2) determine the size of each flight
# 3) chunk each flight at this size
#
def naive_flights(df, num_flights):

    # the standard maximum size for flights is 15 lifters
    MAX_SIZE = 15
    
    # sort the df
    sorted = df.copy().sort_values(by='ProjectedTotal')

    # determine how big each flight will be
    num_lifters = len(df)
    base_size = num_lifters // num_flights
    remainder = num_lifters % num_flights

    if (base_size > MAX_SIZE) or (base_size == MAX_SIZE and remainder > 0):
        raise ValueError("Meet registration is over capacity")

    # create the list of lifter IDs
    ids = sorted.index.to_list()

    flights = []
    index = 0
    for i in range(num_flights):

        flight_size = base_size
        if remainder > 0:
            flight_size += 1
            remainder -= 1

        flights.append(ids[index:index + flight_size])
        index += flight_size

    return flights

In [22]:
def attempt_range(df, column):
    return df[column].max() - df[column].min()

In [56]:
def display_flight_data(flights):
    flight_ranges = {}
    for i, flight in enumerate(flights, start = 1):
        lifters = sample.loc[flight]
        display(lifters)

        lifts = ['Squat1Kg', 'Bench1Kg','Deadlift1Kg']
        lift_ranges = [attempt_range(lifters, lift) for lift in lifts]

        flight_ranges[i] = lift_ranges
    
        print(f"Range for Squat: {lift_ranges[0]}")
        print(f"Range for Bench: {lift_ranges[1]}")
        print(f"Range for Deadlift: {lift_ranges[2]}")

    return flight_ranges

In [57]:
# testing the naive_flights function using the sample data
sample_naive_flights = naive_flights(sample, 4)
print(sample_naive_flights)

display_flight_data(sample_naive_flights)

[[944, 275, 1860, 1083, 674, 1664, 1609, 1333, 128, 1290, 450], [1927, 65, 938, 56, 572, 905, 1981, 1292, 1078, 374, 124], [254, 1289, 1922, 353, 1080, 1179, 1323, 792, 907, 1118], [1646, 964, 1852, 99, 746, 628, 1731, 584, 29, 1273]]


,Name,Sex,Event,Equipment,Division,WeightClassKg,Squat1Kg,Bench1Kg,Deadlift1Kg,ProjectedTotal
944,Iryna Mazur,F,SBD,Single-ply,Sub-Juniors,56,55.0,35.0,80.0,170.0
275,Nanna Nørgaard,F,SBD,Raw,Open,63,70.0,45.0,85.0,200.0
1860,Elisabeth Castleman,F,SBD,Raw,FR-O,57,80.0,42.5,92.5,215.0
1083,Aliya Valieva,F,SBD,Raw,Juniors,57,85.0,47.5,120.0,252.5
674,Morgan Rauls,F,SBD,Raw,FR-Jr,69,87.5,47.5,117.5,252.5
1664,Haley Sharp,F,SBD,Raw,FR-T1,57,120.0,52.5,92.5,265.0
1609,Elizabeth Lubeck,F,SBD,Raw,FR-T2,84,104.3,52.2,115.7,272.2
1333,Christina Paugh,F,SBD,Raw,FR-O,84+,105.0,57.5,110.0,272.5
128,Dunja Klarić,F,SBD,Raw,Masters 1,76,97.5,55.0,120.0,272.5
1290,Silje Møglestue Jacobsen,F,SBD,Raw,Open,84+,115.0,50.0,130.0,295.0


Range for Squat: 65.0
Range for Bench: 30.0
Range for Deadlift: 60.0


,Name,Sex,Event,Equipment,Division,WeightClassKg,Squat1Kg,Bench1Kg,Deadlift1Kg,ProjectedTotal
1927,Jairus Jahner,M,SBD,Raw,MR-JV,74,125.0,75.0,125.0,325.0
65,Dominika Piskorz,F,SBD,Raw,Open,63,120.0,65.0,160.0,345.0
938,Piotr Szlufik,M,SBD,Raw,Juniorzy do lat 20,74,100.0,100.0,150.0,350.0
56,Gérald Thorron,M,SBD,Raw,Masters 2,74,117.5,77.5,160.0,355.0
572,Sybille Hampel,F,SBD,Single-ply,Masters 1,72,145.0,75.0,135.0,355.0
905,Magdolna Petróczki,F,SBD,Single-ply,Open,56,145.0,75.0,160.0,380.0
1981,Atle Aamelfot,M,SBD,Raw,Masters 60-69,105,140.0,90.0,150.0,380.0
1292,Jasmin Higgs,F,SBD,Single-ply,Open,63,145.0,100.0,140.0,385.0
1078,Félix Ricardo Botello Urrutia,M,SBD,Raw,Open,66,130.0,87.5,170.0,387.5
374,Matthew Richards #2,M,SBD,Raw,Juniors,83,145.0,100.0,160.0,405.0


Range for Squat: 45.0
Range for Bench: 35.0
Range for Deadlift: 70.0


,Name,Sex,Event,Equipment,Division,WeightClassKg,Squat1Kg,Bench1Kg,Deadlift1Kg,ProjectedTotal
254,Édouard LeBlond,M,SBD,Raw,Juniors,93,172.5,115.0,180.0,467.5
1289,佐藤 駿,M,SBD,Raw,High School,74,160.0,100.0,207.5,467.5
1922,Eric Whitehead,M,SBD,Raw,MR-O,74,165.0,112.5,200.0,477.5
353,Harry Smith,M,SBD,Raw,MR-Sj,74,170.0,130.0,180.0,480.0
1080,Doug Benedict,M,SBD,Raw,MR-M2a,105,177.5,110.0,207.5,495.0
1179,Niniola Olagundoye,M,SBD,Raw,MR-Sj,120,185.0,135.0,190.0,510.0
1323,Klaus Griesch,M,SBD,Single-ply,Masters 2,74,185.0,160.0,175.0,520.0
792,Toni Saarnio,M,SBD,Raw,Over 40,83,175.0,115.0,240.0,530.0
907,Seth Butler,M,SBD,Raw,MR-T3,83,190.0,120.0,220.0,530.0
1118,Frank Hawley,M,SBD,Raw,MR-O,83,185.0,130.0,230.0,545.0


Range for Squat: 30.0
Range for Bench: 60.0
Range for Deadlift: 65.0


,Name,Sex,Event,Equipment,Division,WeightClassKg,Squat1Kg,Bench1Kg,Deadlift1Kg,ProjectedTotal
1646,Nick Cornish,M,SBD,Raw,Open,93,190.0,120.0,240.0,550.0
964,Mathias Sabiaux,M,SBD,Raw,Masters 1,105,212.5,122.5,230.0,565.0
1852,Célian Marchandise,M,SBD,Raw,Juniors,74,215.0,125.0,235.0,575.0
99,Ravem Noguerraza,M,SBD,Raw,MR-O,93,215.0,130.0,240.0,585.0
746,Liane Blyn,F,SBD,Single-ply,Open,84,220.0,162.5,205.0,587.5
628,Leonard King,M,SBD,Raw,Open,74,215.0,125.0,250.0,590.0
1731,Carlos Mata #1,M,SBD,Raw,MR-O,74,220.0,125.0,255.0,600.0
584,Turanga Rongo,M,SBD,Raw,Open,120+,240.0,155.0,225.0,620.0
29,Dustin Wang,M,SBD,Raw,MR-Uni,105,260.0,142.5,270.0,672.5
1273,Mikael Hedesund,M,SBD,Single-ply,Masters 1,110,295.0,192.5,285.0,772.5


Range for Squat: 105.0
Range for Bench: 72.5
Range for Deadlift: 80.0


{1: [65.0, 30.0, 60.0],
 2: [45.0, 35.0, 70.0],
 3: [30.0, 60.0, 65.0],
 4: [105.0, 72.5, 80.0]}

We can now see the ranges for each of the lifts for each of the flights created by the naive_flights function.

---

In [53]:
# this function creates flights using the KMeans clustering model
#
def kmeans_flights(df, num_flights, scale = True):

    result = df.copy()
    
    # the standard maximum size for flights is 15 lifters
    MAX_SIZE = 15
    
    # select the 1st attempt columns to use for clustering
    features = ['Squat1Kg', 'Bench1Kg', 'Deadlift1Kg']
    X = df[features]

    # scale data if necessary
    if scale:
        scaler = StandardScaler()
        X[features] = scaler.fit_transform(X)
    
    # determine the size of the flights
    num_lifters = len(df)
    flight_size = num_lifters // num_flights

    # determine the flights
    bkmeans = KMeansConstrained(
        n_clusters=num_flights,
        size_min=flight_size,
        size_max=flight_size + 1,
        random_state=42
    )
    result['flight'] = bkmeans.fit_predict(X)
    
    flight_id_map = result.groupby('flight').groups

    print(flight_id_map)
    flights = {f: ids.tolist() for f, ids in flight_id_map.items()}
    
    return flights

In [58]:
kmeans_flights(sample, 4)

{0: [353, 1289, 1323, 1118, 1922, 1179, 792, 907, 1080, 254], 1: [1860, 1333, 275, 128, 674, 1664, 1083, 944, 450, 1609, 1290], 2: [905, 938, 65, 56, 1292, 374, 1981, 572, 1078, 124, 1927], 3: [1273, 1731, 584, 746, 1646, 1852, 99, 964, 29, 628]}


C:\Users\jackm\AppData\Local\Temp\ipykernel_18548\1768851170.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[features] = scaler.fit_transform(X)


{0: [353, 1289, 1323, 1118, 1922, 1179, 792, 907, 1080, 254],
 1: [1860, 1333, 275, 128, 674, 1664, 1083, 944, 450, 1609, 1290],
 2: [905, 938, 65, 56, 1292, 374, 1981, 572, 1078, 124, 1927],
 3: [1273, 1731, 584, 746, 1646, 1852, 99, 964, 29, 628]}